### 460. LFU Cache

<p>Design and implement a data structure for a <a href="https://en.wikipedia.org/wiki/Least_frequently_used" target="_blank">Least Frequently Used (LFU)</a> cache.</p>

<p>Implement the <code>LFUCache</code> class:</p>

<ul>
	<li><code>LFUCache(int capacity)</code> Initializes the object with the <code>capacity</code> of the data structure.</li>
	<li><code>int get(int key)</code> Gets the value of the <code>key</code> if the <code>key</code> exists in the cache. Otherwise, returns <code>-1</code>.</li>
	<li><code>void put(int key, int value)</code> Update the value of the <code>key</code> if present, or inserts the <code>key</code> if not already present. When the cache reaches its <code>capacity</code>, it should invalidate and remove the <strong>least frequently used</strong> key before inserting a new item. For this problem, when there is a <strong>tie</strong> (i.e., two or more keys with the same frequency), the <strong>least recently used</strong> <code>key</code> would be invalidated.</li>
</ul>

<p>To determine the least frequently used key, a <strong>use counter</strong> is maintained for each key in the cache. The key with the smallest <strong>use counter</strong> is the least frequently used key.</p>

<p>When a key is first inserted into the cache, its <strong>use counter</strong> is set to <code>1</code> (due to the <code>put</code> operation). The <strong>use counter</strong> for a key in the cache is incremented either a <code>get</code> or <code>put</code> operation is called on it.</p>

<p>The functions&nbsp;<code data-stringify-type="code">get</code>&nbsp;and&nbsp;<code data-stringify-type="code">put</code>&nbsp;must each run in <code>O(1)</code> average time complexity.</p>

<p>&nbsp;</p>
<p><strong class="example">Example 1:</strong></p>

<pre>
<strong>Input</strong>
[&quot;LFUCache&quot;, &quot;put&quot;, &quot;put&quot;, &quot;get&quot;, &quot;put&quot;, &quot;get&quot;, &quot;get&quot;, &quot;put&quot;, &quot;get&quot;, &quot;get&quot;, &quot;get&quot;]
[[2], [1, 1], [2, 2], [1], [3, 3], [2], [3], [4, 4], [1], [3], [4]]
<strong>Output</strong>
[null, null, null, 1, null, -1, 3, null, -1, 3, 4]

<strong>Explanation</strong>
// cnt(x) = the use counter for key x
// cache=[] will show the last used order for tiebreakers (leftmost element is  most recent)
LFUCache lfu = new LFUCache(2);
lfu.put(1, 1);   // cache=[1,_], cnt(1)=1
lfu.put(2, 2);   // cache=[2,1], cnt(2)=1, cnt(1)=1
lfu.get(1);      // return 1
                 // cache=[1,2], cnt(2)=1, cnt(1)=2
lfu.put(3, 3);   // 2 is the LFU key because cnt(2)=1 is the smallest, invalidate 2.
&nbsp;                // cache=[3,1], cnt(3)=1, cnt(1)=2
lfu.get(2);      // return -1 (not found)
lfu.get(3);      // return 3
                 // cache=[3,1], cnt(3)=2, cnt(1)=2
lfu.put(4, 4);   // Both 1 and 3 have the same cnt, but 1 is LRU, invalidate 1.
                 // cache=[4,3], cnt(4)=1, cnt(3)=2
lfu.get(1);      // return -1 (not found)
lfu.get(3);      // return 3
                 // cache=[3,4], cnt(4)=1, cnt(3)=3
lfu.get(4);      // return 4
                 // cache=[4,3], cnt(4)=2, cnt(3)=3
</pre>

<p>&nbsp;</p>
<p><strong>Constraints:</strong></p>

<ul>
	<li><code>1 &lt;= capacity&nbsp;&lt;= 10<sup>4</sup></code></li>
	<li><code>0 &lt;= key &lt;= 10<sup>5</sup></code></li>
	<li><code>0 &lt;= value &lt;= 10<sup>9</sup></code></li>
	<li>At most <code>2 * 10<sup>5</sup></code>&nbsp;calls will be made to <code>get</code> and <code>put</code>.</li>
</ul>

<p>&nbsp;</p>
<span style="display: none;">&nbsp;</span>

#### Difficulty: Hard, AC rate: 50.4%

#### Topics:
Hash Table | Linked List | Design | Doubly-Linked List

#### Links:
 🎁 [Question Detail](https://leetcode.com/problems/lfu-cache/description/) | 🎉 [Question Solution](https://leetcode.com/problems/lfu-cache/solution/) | 💬 [Question Discussion](https://leetcode.com/problems/lfu-cache/discuss/?orderBy=most_votes)



#### Test Case(s):

1. `["LFUCache","put","put","get","put","get","get","put","get","get","get"]`  
`[[2],[1,1],[2,2],[1],[3,3],[2],[3],[4,4],[1],[3],[4]]`

---
What's your idea?

Double linked list for slots, each slot for a different counter, ordered increasingly.

For each slot, another double linked list for LRU, just like [lru-cache](https://leetcode.com/problems/lru-cache/description/)

---

In [21]:
class Node:
    def __init__(self, address, pre: Node = None, nxt: Node = None):
        self.address = address
        self.pre = pre
        self.nxt = nxt

class SlotNode:
    def __init__(self, cnt: int, pre: SlotNode, nxt: SlotNode, head: Node = None, tail: Node = None):
        self.cnt = cnt
        self.pre = pre
        self.nxt = nxt
        self.head = head
        self.tail = tail
        
class LFUCache:

    def __init__(self, capacity: int):
        self.cache = {'head': [None, 'head', None, None, None], 'tail': [None, 'tail', None, None, None]} # key -> [value, key, cnt, slot_node, node]
        self.slot_start = SlotNode(0, None, None)
        self.slot_end = SlotNode(0, None, None)
        self.slot_start.nxt, self.slot_end.pre = self.slot_end, self.slot_start
        self.capacity = capacity
        self.count = 0

    def __remove_node(self, node: Node):
        node.pre.nxt = node.nxt
        node.nxt.pre = node.pre

    def __check_empty_slot(self, slot_node: SlotNode):
        if slot_node.head.nxt.address[1] == 'tail': # slot's head meets tail, slot is empty
            slot_node.pre.nxt = slot_node.nxt
            slot_node.nxt.pre = slot_node.pre

    def __insert_slot_node(self, slot_node: SlotNode):
        head_node, tail_node = Node(self.cache['head']), Node(self.cache['tail'])
        head_node.nxt, tail_node.pre = tail_node, head_node
        new_slot_node = SlotNode(slot_node.cnt + 1, slot_node, slot_node.nxt, head_node, tail_node)
        slot_node.nxt.pre = new_slot_node
        slot_node.nxt = new_slot_node

    def __insert_node_to_slot(self, node: Node, slot_node: SlotNode):
        tail = slot_node.tail
        node.pre = tail.pre
        node.pre.nxt = node
        node.nxt = tail
        node.nxt.pre = node
        node.address[3] = slot_node

    def __access_key(self, key, is_new: bool = False):
        node = self.cache[key][4]
        slot_node = self.cache[key][3]
        if slot_node.nxt is None or slot_node.nxt.cnt != slot_node.cnt + 1:
            self.__insert_slot_node(slot_node)
        if not is_new:
            self.__remove_node(node)
            self.__check_empty_slot(slot_node)
        slot_node = slot_node.nxt
        self.__insert_node_to_slot(node, slot_node)
        self.cache[key][2] = self.cache[key][2] + 1

    def __evict(self):
        slot_node = self.slot_start.nxt
        head = slot_node.head
        lru_node = head.nxt
        self.__remove_node(lru_node)
        self.count -= 1
        self.__check_empty_slot(slot_node)
        lru_e = lru_node.address
        del self.cache[lru_e[1]]

    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        self.__access_key(key)
        return self.cache[key][0]

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self.cache[key][0] = value
            self.__access_key(key)
            return
        if self.count >= self.capacity:
           self.__evict()
        self.cache[key] = [value, key, 1, self.slot_start, None]
        self.cache[key][4] = Node(self.cache[key])
        self.__access_key(key, True)
        self.count += 1

In [22]:
# Your LFUCache object will be instantiated and called as such:
# obj = LFUCache(capacity)
# param_1 = obj.get(key)
# obj.put(key,value)
obj = LFUCache(2)
for func, params in zip(["put","put","get","put","get","get","put","get","get","get"], [[1,1],[2,2],[1],[3,3],[2],[3],[4,4],[1],[3],[4]]):
    print(f'{func}({params}): {getattr(obj, func)(*params)}')
    # [null, null, null, 1, null, -1, 3, null, -1, 3, 4]

put([1, 1]): None
put([2, 2]): None
get([1]): 1
put([3, 3]): None
get([2]): -1
get([3]): 3
put([4, 4]): None
get([1]): -1
get([3]): 3
get([4]): 4


In [23]:
obj = LFUCache(1)
for func, params in zip(["put","get","put","get","get"], [[2,1],[2],[3,2],[2],[3]]):
    print(f'{func}({params}): {getattr(obj, func)(*params)}')
    # [null,null,1,null,-1,2]

put([2, 1]): None
get([2]): 1
put([3, 2]): None
get([2]): -1
get([3]): 2


In [24]:
obj = LFUCache(10)
for func, params in zip(["put","put","put","put","put","get","put","get","get","put","get","put","put","put","get","put","get","get","get","get","put","put","get","get","get","put","put","get","put","get","put","get","get","get","put","put","put","get","put","get","get","put","put","get","put","put","put","put","get","put","put","get","put","put","get","put","put","put","put","put","get","put","put","get","put","get","get","get","put","get","get","put","put","put","put","get","put","put","put","put","get","get","get","put","put","put","get","put","put","put","get","put","put","put","get","get","get","put","put","put","put","get","put","put","put","put","put","put","put"],
                        [[10,13],[3,17],[6,11],[10,5],[9,10],[13],[2,19],[2],[3],[5,25],[8],[9,22],[5,5],[1,30],[11],[9,12],[7],[5],[8],[9],[4,30],[9,3],[9],[10],[10],[6,14],[3,1],[3],[10,11],[8],[2,14],[1],[5],[4],[11,4],[12,24],[5,18],[13],[7,23],[8],[12],[3,27],[2,12],[5],[2,9],[13,4],[8,18],[1,7],[6],[9,29],[8,21],[5],[6,30],[1,12],[10],[4,15],[7,22],[11,26],[8,17],[9,29],[5],[3,4],[11,30],[12],[4,29],[3],[9],[6],[3,4],[1],[10],[3,29],[10,28],[1,20],[11,13],[3],[3,12],[3,8],[10,9],[3,26],[8],[7],[5],[13,17],[2,27],[11,15],[12],[9,19],[2,15],[3,16],[1],[12,17],[9,1],[6,19],[4],[5],[5],[8,1],[11,7],[5,2],[9,28],[1],[2,2],[7,4],[4,22],[7,24],[9,26],[13,28],[11,26]]):
    print(f'{func}({params}): {getattr(obj, func)(*params)}')
    # [null,null,null,null,null,null,-1,null,19,17,null,-1,null,null,null,-1,null,-1,5,-1,12,null,null,3,5,5,null,null,1,null,-1,null,30,5,30,null,null,null,-1,null,-1,24,null,null,18,null,null,null,null,14,null,null,18,null,null,11,null,null,null,null,null,18,null,null,-1,null,4,29,30,null,12,11,null,null,null,null,29,null,null,null,null,17,-1,18,null,null,null,-1,null,null,null,20,null,null,null,29,18,18,null,null,null,null,20,null,null,null,null,null,null,null]

put([10, 13]): None
put([3, 17]): None
put([6, 11]): None
put([10, 5]): None
put([9, 10]): None
get([13]): -1
put([2, 19]): None
get([2]): 19
get([3]): 17
put([5, 25]): None
get([8]): -1
put([9, 22]): None
put([5, 5]): None
put([1, 30]): None
get([11]): -1
put([9, 12]): None
get([7]): -1
get([5]): 5
get([8]): -1
get([9]): 12
put([4, 30]): None
put([9, 3]): None
get([9]): 3
get([10]): 5
get([10]): 5
put([6, 14]): None
put([3, 1]): None
get([3]): 1
put([10, 11]): None
get([8]): -1
put([2, 14]): None
get([1]): 30
get([5]): 5
get([4]): 30
put([11, 4]): None
put([12, 24]): None
put([5, 18]): None
get([13]): -1
put([7, 23]): None
get([8]): -1
get([12]): 24
put([3, 27]): None
put([2, 12]): None
get([5]): 18
put([2, 9]): None
put([13, 4]): None
put([8, 18]): None
put([1, 7]): None
get([6]): 14
put([9, 29]): None
put([8, 21]): None
get([5]): 18
put([6, 30]): None
put([1, 12]): None
get([10]): 11
put([4, 15]): None
put([7, 22]): None
put([11, 26]): None
put([8, 17]): None
put([9, 29]): None
get(

😃 Result: Accepted  
💯 Passed Test Case: 25 / 25  
🚀 Runtime: 172 ms, Memory: 103.1 MB  
🉑 Runtime Percentile: better than 26.14%, Memory Percentile: better than 5.28%  
📆 Finished At: Mon, 05 Oct 2026 18:14:04 GMT